# Evolution trace

Este notebook simula iteraciones de diseño manuales para mostrar cómo cambian las métricas a lo largo de versiones del prompt.

In [ ]:
from __future__ import annotations

from typing import List

from src.evaluation.dataset import load_dataset
from src.evaluation.runner import run_evaluation
from src.models.agent_config import AgentConfig, FlowStep, ToolSpec


def build_config(version: int, focus: str) -> AgentConfig:
    return AgentConfig(
        system_prompt=(
            "You are an expert Python reviewer. "
            f"Focus on {focus}. Provide concise issue list."
        ),
        tools=[ToolSpec(name="code_analysis", description="Analyze structure."),],
        flow=[FlowStep(name="review", instructions="Review and list issues."),],
        version=version,
    )


dataset = load_dataset("data/benchmark/samples.jsonl")

configs: List[AgentConfig] = [
    build_config(1, "logic bugs"),
    build_config(2, "logic bugs and security"),
    build_config(3, "logic, security, and maintainability"),
]

results = []
for config in configs:
    evaluation = run_evaluation(config, dataset)
    results.append((config.version, evaluation))

print("Evolution trace:")
for version, evaluation in results:
    f1 = next(metric.value for metric in evaluation.metrics if metric.name == "f1")
    precision = next(metric.value for metric in evaluation.metrics if metric.name == "precision")
    recall = next(metric.value for metric in evaluation.metrics if metric.name == "recall")
    print(f"v{version}: f1={f1:.3f}, precision={precision:.3f}, recall={recall:.3f}")

results